# 02 — Limpieza trazable y base analítica con UBIGEO

Este cuaderno transforma la fuente cruda sin sobrescribirla. Implementa el cruce geográfico solicitado y crea dos bases de análisis: una histórica (2020–2025) con variables comparables y otra reciente (2024–2025) que permite variables incorporadas en años recientes.

> No imputa, elimina duplicados ni recodifica targets. Esas acciones requieren evidencia y una decisión posterior documentada.

## Requisitos y resultados

El cuaderno utiliza `BD_2020-2025.csv`, `ubigeo_trabajar.csv` y el perfil generado en `salidas_eda/`. Al activar la exportación crea archivos Parquet en `salidas_limpieza/`; pueden ocupar espacio considerable. La fuente cruda no se modifica.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / 'BD_2020-2025.csv'
UBIGEO_PATH = BASE_DIR / 'ubigeo_trabajar.csv'
EDA_DIR = BASE_DIR / 'salidas_eda'
OUTPUT_DIR = BASE_DIR / 'salidas_limpieza'
OUTPUT_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 100_000
DATE_FORMAT = '%m/%d/%Y'  # Confirmado como hipótesis en EDA; contrastar con ficha de la fuente si existe.
MIN_COVERAGE_HISTORICA = 99.5
MIN_COVERAGE_RECIENTE = 80.0
EJECUTAR_EXPORTACION = True  # Cambiar a True solo después de revisar las celdas de validación.

for path in [DATA_PATH, UBIGEO_PATH, EDA_DIR / 'cobertura_variables_muestra_por_anio.csv']:
    assert path.exists(), f'Falta el insumo requerido: {path.name}'
print('Insumos verificados. Exportación activa:', EJECUTAR_EXPORTACION)

Insumos verificados. Exportación activa: True


## 1. Maestro geográfico y contrato del cruce

El UBIGEO peruano se representa como seis caracteres: departamento (2), provincia (2) y distrito (2). Las columnas originales se conservan; `ubigeo_codigo` es una clave derivada para el cruce y `ubigeo_nombre` es una etiqueta legible.

In [2]:
def limpiar_codigo(serie: pd.Series, ancho: int) -> pd.Series:
    """Convierte códigos a texto de ancho fijo sin alterar la columna fuente."""
    texto = serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    valido = texto.str.fullmatch(r'\d{1,' + str(ancho) + r'}', na=False)
    return texto.where(valido).str.zfill(ancho)

ubigeo = pd.read_csv(UBIGEO_PATH, dtype='string')
ubigeo.columns = ubigeo.columns.str.strip()
required_master = {'Codigo_dpto', 'dpto', 'Codigo_prov', 'Prov', 'Codigo_dist', 'dist'}
assert required_master.issubset(ubigeo.columns), 'El maestro UBIGEO no tiene la estructura esperada.'

ubigeo['ubigeo_codigo'] = (
    limpiar_codigo(ubigeo['Codigo_dpto'], 2)
    + limpiar_codigo(ubigeo['Codigo_prov'], 2)
    + limpiar_codigo(ubigeo['Codigo_dist'], 2)
)
ubigeo['ubigeo_nombre'] = (
    ubigeo['dist'].str.strip() + ', ' + ubigeo['Prov'].str.strip() + ', ' + ubigeo['dpto'].str.strip()
)
ubigeo_lookup = ubigeo[['ubigeo_codigo', 'ubigeo_nombre']].dropna().copy()
assert ubigeo_lookup['ubigeo_codigo'].is_unique, 'El maestro contiene UBIGEO duplicados; detener y revisar.'
print(f'UBIGEO únicos en maestro: {len(ubigeo_lookup):,}')
display(ubigeo_lookup.head())

UBIGEO únicos en maestro: 1,892


,ubigeo_codigo,ubigeo_nombre
0,010101,"Chachapoyas, Chachapoyas, Amazonas"
1,010102,"Asunción, Chachapoyas, Amazonas"
2,010103,"Balsas, Chachapoyas, Amazonas"
3,010104,"Cheto, Chachapoyas, Amazonas"
4,010105,"Chiliquin, Chachapoyas, Amazonas"


In [3]:
raw_columns = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
geo_source_columns = ['DPTO_DOMICILIO', 'PROV_DOMICILIO', 'DIST_DOMICILIO']
assert set(geo_source_columns).issubset(raw_columns), 'No se encontraron las tres columnas geográficas de la fuente.'

def agregar_ubigeo(df: pd.DataFrame) -> pd.DataFrame:
    resultado = df.copy()
    resultado['ubigeo_codigo'] = (
        limpiar_codigo(resultado['DPTO_DOMICILIO'], 2)
        + limpiar_codigo(resultado['PROV_DOMICILIO'], 2)
        + limpiar_codigo(resultado['DIST_DOMICILIO'], 2)
    )
    return resultado.merge(ubigeo_lookup, on='ubigeo_codigo', how='left', validate='many_to_one')

muestra_geo = pd.read_csv(DATA_PATH, dtype='string', nrows=100_000, low_memory=False)
muestra_geo = agregar_ubigeo(muestra_geo)
tasa_match_muestra = muestra_geo['ubigeo_nombre'].notna().mean() * 100
print(f'Coincidencia UBIGEO en muestra de 100 000 filas: {tasa_match_muestra:.2f}%')
display((muestra_geo.loc[muestra_geo['ubigeo_nombre'].isna(), geo_source_columns + ['ubigeo_codigo']]
         .value_counts(dropna=False).head(20).rename('n').reset_index()))

Coincidencia UBIGEO en muestra de 100 000 filas: 100.00%


,DPTO_DOMICILIO,PROV_DOMICILIO,DIST_DOMICILIO,ubigeo_codigo,n
0,99,99,99,999999,3


## 2. Definir escenarios con la evidencia del EDA

La cobertura se obtiene del EDA por año. El escenario histórico exige al menos 99.5% de cobertura en cada año; el reciente exige 80% en 2024 y 2025. Los umbrales son parámetros explícitos, no reglas ocultas.

In [4]:
coverage = pd.read_csv(EDA_DIR / 'cobertura_variables_muestra_por_anio.csv')
year_columns = [str(year) for year in range(2020, 2026)]
assert set(year_columns).issubset(coverage.columns), 'El reporte de cobertura no incluye todos los años esperados.'

core_columns = ['CEM', 'FECHA_INGRESO', 'EDAD_VICTIMA', 'SEXO_VICTIMA', 'TIPO_VIOLENCIA',
                'NIVEL_DE_RIESGO_VICTIMA', *geo_source_columns]
historical_from_coverage = coverage.loc[coverage[year_columns].min(axis=1) >= MIN_COVERAGE_HISTORICA, 'columna_original'].tolist()
recent_from_coverage = coverage.loc[coverage[['2024', '2025']].min(axis=1) >= MIN_COVERAGE_RECIENTE, 'columna_original'].tolist()

historical_columns = list(dict.fromkeys([*core_columns, *historical_from_coverage, 'ubigeo_codigo', 'ubigeo_nombre']))
recent_columns = list(dict.fromkeys([*core_columns, *recent_from_coverage, 'ubigeo_codigo', 'ubigeo_nombre']))

assert set(core_columns).issubset(historical_columns)
print(f'Variables base histórica: {len(historical_columns)}')
print(f'Variables base reciente: {len(recent_columns)}')
print('MODALIDADES_VCM en histórica:', 'MODALIDADES_VCM' in historical_columns)
print('MODALIDADES_VCM en reciente:', 'MODALIDADES_VCM' in recent_columns)

Variables base histórica: 33
Variables base reciente: 55
MODALIDADES_VCM en histórica: False
MODALIDADES_VCM en reciente: True


## 3. Exportación por bloques

Esta celda solo opera al cambiar `EJECUTAR_EXPORTACION=True`. Lee una vez el CSV, cruza UBIGEO, separa los dos escenarios y escribe Parquet por bloques. Reemplaza únicamente sus archivos de salida para que la ejecución sea reproducible.

In [5]:
if EJECUTAR_EXPORTACION:
    import pyarrow as pa
    import pyarrow.parquet as pq

    historic_path = OUTPUT_DIR / 'base_historica_2020_2025_ubigeo.parquet'
    recent_path = OUTPUT_DIR / 'base_reciente_2024_2025_ubigeo.parquet'
    for path in [historic_path, recent_path]:
        if path.exists():
            path.unlink()

    writers = {'historica': None, 'reciente': None}
    summary = []
    for chunk_number, chunk in enumerate(pd.read_csv(DATA_PATH, dtype='string', chunksize=CHUNK_SIZE, low_memory=False), start=1):
        chunk = agregar_ubigeo(chunk)
        fechas = pd.to_datetime(chunk['FECHA_INGRESO'], format=DATE_FORMAT, errors='coerce')
        chunk['anio_ingreso'] = fechas.dt.year.astype('Int64')
        historica = chunk.loc[chunk['anio_ingreso'].between(2020, 2025, inclusive='both'), historical_columns + ['anio_ingreso']]
        reciente = chunk.loc[chunk['anio_ingreso'].isin([2024, 2025]), recent_columns + ['anio_ingreso']]

        for nombre, tabla, path in [('historica', historica, historic_path), ('reciente', reciente, recent_path)]:
            arrow_table = pa.Table.from_pandas(tabla, preserve_index=False)
            if writers[nombre] is None:
                writers[nombre] = pq.ParquetWriter(path, arrow_table.schema, compression='snappy')
            writers[nombre].write_table(arrow_table)

        summary.append({
            'chunk': chunk_number, 'filas_leidas': len(chunk),
            'match_ubigeo_pct': round(chunk['ubigeo_nombre'].notna().mean() * 100, 4),
            'fechas_invalidas': int(fechas.isna().sum()),
            'filas_historica': len(historica), 'filas_reciente': len(reciente),
        })

    for writer in writers.values():
        if writer is not None:
            writer.close()

    export_summary = pd.DataFrame(summary)
    export_summary.to_csv(OUTPUT_DIR / 'resumen_exportacion_y_calidad.csv', index=False, encoding='utf-8-sig')
    print('Exportación terminada.')
    display(export_summary.agg({'filas_leidas': 'sum', 'filas_historica': 'sum', 'filas_reciente': 'sum', 'fechas_invalidas': 'sum', 'match_ubigeo_pct': 'mean'}))
else:
    print('Exportación no ejecutada. Revise el match UBIGEO y active EJECUTAR_EXPORTACION para crear las bases.')

Exportación terminada.


filas_leidas        936835.00000
filas_historica     936835.00000
filas_reciente      338028.00000
fechas_invalidas         0.00000
match_ubigeo_pct        99.98941
dtype: float64

## 4. Registro de decisiones

La exportación no es un dataset de modelado. Antes de entrenar un modelo se debe fijar un objetivo, fecha de corte y exclusiones por fuga de información. Esta bitácora documenta lo acordado en esta fase.

In [6]:
decisions = pd.DataFrame([
    ['UBIGEO', 'Crear ubigeo_codigo y ubigeo_nombre mediante maestro validado.', 'Preserva DPTO/PROV/DIST originales; mide match y no inventa ubicaciones.', 'Aprobada'],
    ['Escenario histórico', f'Incluir variables con >= {MIN_COVERAGE_HISTORICA}% de cobertura anual en 2020–2025.', 'Evita comparar variables no capturadas de modo consistente.', 'Aprobada'],
    ['Escenario reciente', f'Incluir variables con >= {MIN_COVERAGE_RECIENTE}% de cobertura en 2024–2025.', 'Permite analizar variables nuevas sin proyectarlas hacia atrás.', 'Aprobada'],
    ['Nulos', 'No imputar en esta etapa.', 'Un nulo puede ser no aplica, no capturado o faltante real.', 'Pendiente de diccionario'],
    ['Modelado', 'No seleccionar features ni targets aquí.', 'Requiere prevención explícita de fuga y split temporal.', 'Pendiente'],
], columns=['tema', 'decision', 'justificacion', 'estado'])
decisions.to_csv(OUTPUT_DIR / 'registro_decisiones_limpieza.csv', index=False, encoding='utf-8-sig')
display(decisions)

,tema,decision,justificacion,estado
0,UBIGEO,Crear ubigeo_codigo y ubigeo_nombre mediante m...,Preserva DPTO/PROV/DIST originales; mide match...,Aprobada
1,Escenario histórico,Incluir variables con >= 99.5% de cobertura an...,Evita comparar variables no capturadas de modo...,Aprobada
2,Escenario reciente,Incluir variables con >= 80.0% de cobertura en...,Permite analizar variables nuevas sin proyecta...,Aprobada
3,Nulos,No imputar en esta etapa.,"Un nulo puede ser no aplica, no capturado o fa...",Pendiente de diccionario
4,Modelado,No seleccionar features ni targets aquí.,Requiere prevención explícita de fuga y split ...,Pendiente


## Siguiente fase

Tras revisar el porcentaje de match UBIGEO y ejecutar la exportación, el siguiente cuaderno debe definir el objetivo analítico, reglas de valores válidos, tratamiento explícito de nulos y un particionado temporal. Ninguna columna se debe usar como predictor hasta verificar que está disponible antes del resultado que se desea predecir.